In [0]:
pip install lyricsgenius boto3 openai

In [0]:
dbutils.library.restartPython()

In [0]:
import json
import time
import boto3
import lyricsgenius
from openai import OpenAI
from pyspark.sql.functions import col, rand
from datetime import datetime

In [0]:
# Configurar credenciais
GENIUS_TOKEN = dbutils.secrets.get(scope="SPOTIFY", key="CLIENT_ACCESS_TOKEN_GENIUS")
DEEPSEEK_API_KEY = dbutils.secrets.get(scope="SPOTIFY", key="APIKEY_DEEPSEEK")
AWS_ACCESS_KEY = dbutils.secrets.get(scope="AWS_BUCKET", key="AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = dbutils.secrets.get(scope="AWS_BUCKET", key="AWS_SECRET_ACCESS_KEY")

# Inicializar clientes
genius = lyricsgenius.Genius(
    GENIUS_TOKEN,
    timeout=15,
    sleep_time=0.5,
    retries=3
)
# Configurar User-Agent para evitar bloqueio do Cloudflare
genius.response_format = 'plain'
genius.remove_section_headers = True

deepseek_client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name='us-east-1'
)

In [0]:
# Configurações
BUCKET_NAME = "bucketbastet"
S3_PREFIX = "letras/"
LOTE_SIZE = 5
TABELA_MUSICAS = "gold.spotify.spotify_musicas"
TABELA_ANALISE_LETRAS = "gold.spotify.analise_letras"

In [0]:
# Criar tabela de controle se não existir
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABELA_ANALISE_LETRAS} (
    master_metadata_track_name STRING,
    master_metadata_album_artist_name STRING,
    master_metadata_album_album_name STRING,
    genius_url STRING,
    sentimento STRING,
    emocao_primaria STRING,
    temas ARRAY<STRING>,
    idioma STRING,
    tem_letra BOOLEAN,
    confianca_analise FLOAT,
    s3_path STRING,
    data_processamento TIMESTAMP,
    erro STRING
) USING DELTA
""")

print(f"Tabela {TABELA_ANALISE_LETRAS} pronta.")

In [0]:
# Identificar músicas ainda não processadas
df_musicas = spark.table(TABELA_MUSICAS)
df_processadas = spark.table(TABELA_ANALISE_LETRAS)

df_pendentes = (
    df_musicas.alias("m")
    .join(
        df_processadas.alias("p"),
        [
            col("m.master_metadata_track_name") == col("p.master_metadata_track_name"),
            col("m.master_metadata_album_artist_name") == col("p.master_metadata_album_artist_name"),
            col("m.master_metadata_album_album_name") == col("p.master_metadata_album_album_name")
        ],
        "left_anti"
    )
    .select(
        "m.master_metadata_track_name",
        "m.master_metadata_album_artist_name",
        "m.master_metadata_album_album_name"
    )
)

total_pendentes = df_pendentes.count()
print(f"Total de músicas pendentes de análise: {total_pendentes}")

if total_pendentes == 0:
    dbutils.notebook.exit("Nenhuma música nova para processar. Job concluído.")

In [0]:
# Selecionar lote aleatório
df_lote = df_pendentes.orderBy(rand()).limit(LOTE_SIZE)
musicas_lote = [row.asDict() for row in df_lote.collect()]

print(f"Selecionadas {len(musicas_lote)} músicas para processar neste lote.")
for musica in musicas_lote:
    print(f"  - {musica['master_metadata_album_artist_name']} - {musica['master_metadata_track_name']}")

In [0]:
def buscar_letra_genius(artista, titulo):
    """
    Busca a letra no Genius.
    Retorna: (letra_texto, url_genius, sucesso)
    """
    try:
        print(f"Buscando: {artista} - {titulo}")
        song = genius.search_song(titulo, artista)
        
        if song and song.lyrics:
            # Limpar o texto da letra (remover cabeçalhos do Genius)
            letra = song.lyrics
            # Remove padrões como "23Embed" ou "12Contributors"
            import re
            letra = re.sub(r'\d+Embed$', '', letra)
            letra = re.sub(r'\d+Contributors.*?Lyrics', '', letra, flags=re.DOTALL)
            letra = letra.strip()
            
            return letra, song.url, True
        else:
            return None, None, False
            
    except Exception as e:
        erro_str = str(e)
        
        # Tratar erros específicos
        if "403" in erro_str or "Forbidden" in erro_str:
            print(f"  ⚠️ Erro 403 (Cloudflare): A API do Genius está bloqueando temporariamente. Aguardando...")
            time.sleep(3)  # Pausa maior para evitar rate limiting
            return None, None, False
        elif "404" in erro_str or "Not Found" in erro_str:
            print(f"  ℹ️ Música não encontrada no Genius")
            return None, None, False
        elif "timeout" in erro_str.lower():
            print(f"  ⏱️ Timeout ao buscar no Genius")
            return None, None, False
        else:
            print(f"  ❌ Erro ao buscar no Genius: {erro_str[:200]}")
            return None, None, False

In [0]:
def analisar_letra_deepseek(letra, artista, titulo):
    """
    Analisa a letra usando DeepSeek para extrair:
    - Sentimento geral
    - Emoção primária
    - Temas principais
    - Idioma
    - Confiança na análise
    
    Detecta automaticamente músicas instrumentais.
    """
    
    # Tratamento para músicas instrumentais
    letra_limpa = letra.strip() if letra else ""
    
    # Detecta se é instrumental (letra muito curta ou vazia)
    if len(letra_limpa) < 50 or not letra_limpa:
        print("  ⚠️ Música identificada como INSTRUMENTAL (letra ausente ou muito curta)")
        return {
            "sentimento": "neutro",
            "emocao_primaria": "instrumental",
            "temas": ["instrumental"],
            "idioma": "não aplicável",
            "confianca_analise": 1.0,
            "observacoes": "Música instrumental sem letra para análise"
        }
    
    # Detecta padrões comuns de letras não encontradas
    palavras_indicadoras_instrumental = [
        "instrumental",
        "no lyrics",
        "sem letra",
        "without lyrics"
    ]
    
    letra_lower = letra_limpa.lower()
    if any(palavra in letra_lower for palavra in palavras_indicadoras_instrumental):
        print("  ⚠️ Música identificada como INSTRUMENTAL (padrão detectado na letra)")
        return {
            "sentimento": "neutro",
            "emocao_primaria": "instrumental",
            "temas": ["instrumental"],
            "idioma": "não aplicável",
            "confianca_analise": 1.0,
            "observacoes": "Música instrumental sem letra para análise"
        }
    
    system_prompt = """
Você é um especialista em análise de letras de músicas. Analise a letra fornecida e retorne APENAS um JSON válido com a seguinte estrutura:

{
  "sentimento": "positivo" | "negativo" | "neutro" | "misto",
  "emocao_primaria": "alegria" | "tristeza" | "raiva" | "medo" | "amor" | "nostalgia" | "esperança" | "melancolia" | "outro",
  "temas": ["tema1", "tema2", "tema3"],
  "idioma": "português" | "inglês" | "espanhol" | "alemão" | "árabe" | "italiano" | "russo" | "chinês" | "japonês" | "francês" | "coreano" | "hebreu" | "turco" | "outro",
  "confianca_analise": 0.0 a 1.0,
  "observacoes": "Breve comentário sobre a análise"
}

IMPORTANTE:
- Retorne APENAS o JSON, sem markdown ou texto adicional
- Os temas devem ser específicos e relevantes (máximo 3)
- A confiança deve refletir quão clara é a mensagem da letra
- Para letras com conteúdo explícito, analise de forma neutra e profissional
""".strip()
    
    user_content = f"""
Artista: {artista}
Música: {titulo}

Letra:
{letra[:3000]}  # Limitar a 3000 caracteres para economizar tokens
"""
    
    try:
        response = deepseek_client.chat.completions.create(
            model="deepseek-chat",
            temperature=0.1,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ]
        )
        
        analise = json.loads(response.choices[0].message.content)
        return analise
        
    except Exception as e:
        print(f"  ❌ Erro ao analisar com DeepSeek: {str(e)}")
        return {
            "sentimento": "erro",
            "emocao_primaria": "erro",
            "temas": ["erro na análise"],
            "idioma": "desconhecido",
            "confianca_analise": 0.0,
            "observacoes": str(e)
        }

In [0]:
def salvar_em_s3(dados, track_name, artist_name):
    """
    Salva os dados (letra + análise) em S3 como JSON.
    Retorna o caminho S3.
    """
    try:
        # Criar nome de arquivo seguro
        nome_arquivo = f"{artist_name}_{track_name}".lower()
        nome_arquivo = "".join(c if c.isalnum() or c in ('-', '_') else '_' for c in nome_arquivo)
        nome_arquivo = f"{nome_arquivo}_{int(time.time())}.json"
        
        s3_key = f"{S3_PREFIX}{nome_arquivo}"
        
        # Converter para JSON
        json_data = json.dumps(dados, ensure_ascii=False, indent=2)
        
        # Upload para S3
        s3_client.put_object(
            Bucket=BUCKET_NAME,
            Key=s3_key,
            Body=json_data.encode('utf-8'),
            ContentType='application/json'
        )
        
        s3_path = f"s3://{BUCKET_NAME}/{s3_key}"
        print(f"Salvo em: {s3_path}")
        return s3_path
        
    except Exception as e:
        print(f"Erro ao salvar em S3: {str(e)}")
        return None

In [0]:
# Processar cada música do lote
resultados = []

for musica in musicas_lote:
    track_name = musica['master_metadata_track_name']
    artist_name = musica['master_metadata_album_artist_name']
    album_name = musica['master_metadata_album_album_name']
    
    print(f"\n{'='*60}")
    print(f"Processando: {artist_name} - {track_name}")
    print(f"{'='*60}")
    
    resultado = {
        'master_metadata_track_name': track_name,
        'master_metadata_album_artist_name': artist_name,
        'master_metadata_album_album_name': album_name,
        'data_processamento': datetime.now().isoformat()
    }
    
    # 1. Buscar letra no Genius
    letra, genius_url, encontrou = buscar_letra_genius(artist_name, track_name)
    
    if not encontrou or not letra:
        print("❌ Letra não encontrada no Genius")
        resultado.update({
            'genius_url': None,
            'tem_letra': False,
            'sentimento': None,
            'emocao_primaria': None,
            'temas': None,
            'idioma': None,
            'confianca_analise': None,
            's3_path': None,
            'erro': 'Letra não encontrada no Genius'
        })
        resultados.append(resultado)
        continue
    
    print(f"✓ Letra encontrada: {genius_url}")
    resultado['genius_url'] = genius_url
    resultado['tem_letra'] = True
    
    # 2. Analisar com DeepSeek
    print("Analisando sentimentos e temas com DeepSeek...")
    analise = analisar_letra_deepseek(letra, artist_name, track_name)
    
    resultado.update({
        'sentimento': analise.get('sentimento'),
        'emocao_primaria': analise.get('emocao_primaria'),
        'temas': analise.get('temas'),
        'idioma': analise.get('idioma'),
        'confianca_analise': float(analise.get('confianca_analise', 0.0)),
        'erro': None
    })
    
    print(f"  Sentimento: {analise.get('sentimento')}")
    print(f"  Emoção: {analise.get('emocao_primaria')}")
    print(f"  Temas: {', '.join(analise.get('temas', []))}")
    print(f"  Idioma: {analise.get('idioma')}")
    
    # 3. Salvar em S3
    print("Salvando em S3...")
    dados_completos = {
        'metadata': {
            'track_name': track_name,
            'artist_name': artist_name,
            'album_name': album_name,
            'genius_url': genius_url,
            'data_processamento': resultado['data_processamento']
        },
        'letra': letra,
        'analise': analise
    }
    
    s3_path = salvar_em_s3(dados_completos, track_name, artist_name)
    resultado['s3_path'] = s3_path
    
    resultados.append(resultado)
    
    # Pausa entre requisições
    time.sleep(1)

print(f"\n{'='*60}")
print(f"Processamento concluído: {len(resultados)} músicas")
print(f"{'='*60}")

In [0]:
# Salvar resultados na tabela Delta
if resultados:
    from pyspark.sql.types import (
        StructType, StructField, StringType, BooleanType, 
        FloatType, TimestampType, ArrayType
    )
    
    # Definir schema explícito para evitar erro de inferência
    schema = StructType([
        StructField("master_metadata_track_name", StringType(), True),
        StructField("master_metadata_album_artist_name", StringType(), True),
        StructField("master_metadata_album_album_name", StringType(), True),
        StructField("genius_url", StringType(), True),
        StructField("sentimento", StringType(), True),
        StructField("emocao_primaria", StringType(), True),
        StructField("temas", ArrayType(StringType()), True),
        StructField("idioma", StringType(), True),
        StructField("tem_letra", BooleanType(), True),
        StructField("confianca_analise", FloatType(), True),
        StructField("s3_path", StringType(), True),
        StructField("data_processamento", StringType(), True),  # Será convertido para timestamp
        StructField("erro", StringType(), True)
    ])
    
    # Criar DataFrame com schema explícito
    df_resultados = spark.createDataFrame(resultados, schema=schema)
    
    # Converter data_processamento para timestamp
    from pyspark.sql.functions import to_timestamp
    df_resultados = df_resultados.withColumn(
        "data_processamento", 
        to_timestamp(col("data_processamento"))
    )
    
    df_resultados.write.mode("append").saveAsTable(TABELA_ANALISE_LETRAS)
    
    print(f"✓ Resultados salvos na tabela {TABELA_ANALISE_LETRAS}")
    display(df_resultados)

In [0]:
# Estatísticas finais
df_estatisticas = spark.table(TABELA_ANALISE_LETRAS)

print("\n📊 ESTATÍSTICAS GERAIS:")
print(f"Total de músicas processadas: {df_estatisticas.count()}")
print(f"Músicas com letra encontrada: {df_estatisticas.filter(col('tem_letra') == True).count()}")
print(f"Músicas sem letra: {df_estatisticas.filter(col('tem_letra') == False).count()}")

print("\n📈 Distribuição de Sentimentos:")
display(
    df_estatisticas
    .filter(col('sentimento').isNotNull())
    .groupBy('sentimento')
    .count()
    .orderBy(col('count').desc())
)

print("\n🎭 Distribuição de Emoções:")
display(
    df_estatisticas
    .filter(col('emocao_primaria').isNotNull())
    .groupBy('emocao_primaria')
    .count()
    .orderBy(col('count').desc())
)